# Girls — predicting next education progress score (CRISP-DM, Phases 1–6)

**Goal:** for each education check-in, predict a resident's progress score at her *next* visit. Residents whose predicted next score is low relative to peers are flagged for prioritised case-worker attention.

## Six phases used in this notebook

1. **Business understanding** — define the forward-prediction target and the intervention decision it supports.
2. **Data understanding** — inventory longitudinal tables and assess temporal coverage.
3. **Data preparation** — construct one row per consecutive education-record pair (~474 rows); engineer temporally windowed features so no future data leaks into any row.
4. **Modeling** — compare regularised regressors with GroupKFold CV (grouped by resident so no resident appears in both train and test).
5. **Evaluation** — OOF metrics, per-resident predicted-progress roster, feature signals.
6. **Deployment** — save reproducible artifact, summarise next actions.

## Why record-level instead of one-row-per-resident?

Each resident has ~9 education check-ins spanning ~8 months (n = 60 residents → ~474 consecutive-record pairs). Collapsing to one row per resident throws away within-resident temporal variation and leaves n = 60, which is too small for stable CV estimates. Using all record pairs gives ~474 rows while keeping the problem genuinely **predictive** (features at time T → outcome at time T+1).

## Phase 1: Business understanding

**Decision supported:** identify residents whose educational progress is advancing unusually slowly so case teams can prioritise intervention — additional tutoring, health support, reduced incident load, etc.

**Target:** `progress_slope_per_month` — percentage-point gain per month, estimated by OLS regression over each resident's longitudinal education records. A higher value means faster improvement.

**"Slow improver" flag (reporting only):** residents in the bottom quartile of slope. This is used for output tables and staff briefings, not as the model's training target.

**Caution:**
- This analysis identifies *associations*, not causal proof.
- The dataset covers residents who *stayed*; girls who dropped out early are absent (survivorship bias). Results reflect factors associated with improvement speed among persistent residents.
- Results should support staff judgement, not replace it.

**Checkpoint — Phase 1:** confirm the improvement-rate framing is the right decision lens before moving to production.

### Approach: Predictive (not explanatory)

This notebook uses a **predictive** approach, not an explanatory or causal one.

**Goal:** Forecast each resident's next education progress score from her current check-in
record, so case managers can flag residents with slow expected improvement for early
intervention — not to establish *why* progress is slow.

**Why predictive, not explanatory:**
An explanatory model would require interpretable coefficients with confidence intervals on
each feature (e.g. "each additional home visitation is associated with X progress points"),
along with assumptions about feature independence and causal ordering. We use a
gradient-boosted ensemble evaluated by cross-validated MAE because accurate out-of-sample
ranking of residents by predicted progress is more useful to case teams than significance
tests on individual features.

**Why not causal:**
A causal claim — e.g. "assigning more counselling sessions causes faster progress" — would
require a randomised design or an identification strategy that controls for selection into
programme intensity. In this observational data, residents with more sessions may simply
be those who were already at higher risk, making naive comparisons misleading. We use
GroupKFold (grouped by `resident_id`) to ensure the model is evaluated on genuinely
unseen residents, not just unseen time windows for seen residents. Model feature importances
are reported as exploratory signals only, explicitly labelled "not causal proof."

## Phase 2: Data understanding

We inventory tables directly tied to resident education risk signals:

- `residents.csv` (demographics/case profile)
- `education_records.csv` (progress/attendance/status)
- `health_wellbeing_records.csv`
- `intervention_plans.csv`
- `home_visitations.csv`
- `incident_reports.csv`
- `process_recordings.csv`
- `safehouses.csv`

The next cell loads these and confirms record volume.

In [1]:
# Phase 2 — load candidate datasets and inventory
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

_repo = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "Dataset").is_dir()),
    None,
)
if _repo is None:
    raise FileNotFoundError("Could not find Dataset/ — open Jupyter from inside the ml-pipelines repo.")

DATA = _repo / "Dataset" / "lighthouse_csv_v7"

TABLE_PATHS = {
    "residents": DATA / "residents.csv",
    "education_records": DATA / "education_records.csv",
    "health_wellbeing_records": DATA / "health_wellbeing_records.csv",
    "intervention_plans": DATA / "intervention_plans.csv",
    "home_visitations": DATA / "home_visitations.csv",
    "incident_reports": DATA / "incident_reports.csv",
    "process_recordings": DATA / "process_recordings.csv",
    "safehouses": DATA / "safehouses.csv",
}

print("=== Candidate tables for struggle analysis ===")
for name, path in TABLE_PATHS.items():
    n = len(pd.read_csv(path, usecols=[0]))
    print(f"{name:25s} rows={n:4d}  file={path.name}")

res = pd.read_csv(TABLE_PATHS["residents"])
edu = pd.read_csv(TABLE_PATHS["education_records"])
hw = pd.read_csv(TABLE_PATHS["health_wellbeing_records"])
plans = pd.read_csv(TABLE_PATHS["intervention_plans"])
visits = pd.read_csv(TABLE_PATHS["home_visitations"])
incidents = pd.read_csv(TABLE_PATHS["incident_reports"])
proc = pd.read_csv(TABLE_PATHS["process_recordings"])
safehouses = pd.read_csv(TABLE_PATHS["safehouses"])

print("\nPrimary resident grain:", len(res), "residents")
print("Education records per resident (mean):", round(len(edu) / len(res), 2))
print("Health records per resident (mean):", round(len(hw) / len(res), 2))
print("Incident reports per resident (mean):", round(len(incidents) / len(res), 2))
print("\nCheckpoint — Phase 2: tables loaded.")

=== Candidate tables for struggle analysis ===
residents                 rows=  60  file=residents.csv
education_records         rows= 534  file=education_records.csv
health_wellbeing_records  rows= 534  file=health_wellbeing_records.csv
intervention_plans        rows= 180  file=intervention_plans.csv
home_visitations          rows=1337  file=home_visitations.csv
incident_reports          rows= 100  file=incident_reports.csv
process_recordings        rows=2819  file=process_recordings.csv
safehouses                rows=   9  file=safehouses.csv

Primary resident grain: 60 residents
Education records per resident (mean): 8.9
Health records per resident (mean): 8.9
Incident reports per resident (mean): 1.67

Checkpoint — Phase 2: tables loaded.


### Phase 2 — Outlier detection
IQR-based outlier check on key numeric features from the modeling frame. Flagged rows are
documented but **not removed** — tree-based models (RandomForest, GradientBoosting) are
naturally robust to covariate outliers, and extreme values in incident counts or health
scores reflect genuine variation in resident circumstances, not data entry errors.

In [ ]:
# Phase 2 — Outlier detection (IQR method, exploratory only)
# Run after the modeling frame (df or modeling_df) is built in the cell above.
# Adjust the variable name below if the frame has a different name.
_frame = None
for _name in ["modeling_df", "df_model", "df", "records"]:
    if _name in dir():
        _frame = eval(_name)
        print(f"Using frame: {_name!r}  shape={_frame.shape}")
        break

_outlier_cols = [
    "n_incidents", "n_home_visitations", "n_process_sessions",
    "n_intervention_plans", "hw_mean_bmi", "hw_mean_nutrition_score",
    "current_progress", "days_since_admission", "days_to_next_record",
]

if _frame is not None:
    import numpy as np
    print(f"\n{'Column':<40} {'Q1':>8} {'Q3':>8} {'IQR':>8}  {'Fence [lo, hi]':<22}  {'n_outliers':>10}")
    print("-" * 105)
    for col in _outlier_cols:
        if col not in _frame.columns:
            continue
        s = _frame[col].dropna()
        Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
        IQR = Q3 - Q1
        lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_out = int(((s < lo) | (s > hi)).sum())
        print(f"{col:<40} {Q1:>8.2f} {Q3:>8.2f} {IQR:>8.2f}  [{lo:>8.2f}, {hi:>8.2f}]  {n_out:>10d}")
    print()
    print("Decision: Leave all outliers.")
    print("  Extreme incident/session counts reflect real high-need cases — not noise.")
    print("  Tree models used in Phase 4 are robust to covariate outliers.")
else:
    print("Frame not found — run the data-building cell above first.")

In [2]:
# ## Phase 3: Data preparation
#
# Row construction (one row per consecutive education-record pair):
#   For each resident, sort records by date. For records i = 0…N-2:
#     - current_progress    = progress_percent at record i
#     - progress_delta      = current_progress minus previous_progress (NaN for i=0)
#     - n_prior_edu_records = i (engagement depth; 0 for first record)
#     - next_progress       = progress_percent at record i+1  ← TARGET
#     - edu_education_level = education_level at record i
#
# Temporal windowing (no future leakage):
#   All aggregated features (health, incidents, visits, process sessions,
#   attendance) are computed from records dated ≤ record_date of row i.
#   This mirrors what a case worker would actually know at each check-in.
#
# Static features joined once per resident: demographics, safehouse context.
#   NOTE: only truly static fields are used (initial_risk_level, case_category,
#   referral_source). Dynamic fields (current_risk_level, case_status,
#   reintegration_status) are excluded to avoid leaking future state into early rows.
#
# Output: frame (~474 rows), X_traj, y_traj, groups (resident_id array for GroupKFold)

In [3]:
# Phase 3 — build record-level modeling frame with temporally windowed features
import re

import numpy as np
import pandas as pd


def parse_years_months(value):
    if pd.isna(value) or not isinstance(value, str):
        return np.nan
    s = value.strip()
    m = re.match(r"(\d+)\s*[Yy]ears?\s*(\d+)\s*[Mm]onths?", s)
    if m:
        return int(m.group(1)) + int(m.group(2)) / 12.0
    m2 = re.match(r"(\d+)\s*[Yy]ears?", s)
    if m2:
        return float(m2.group(1))
    return np.nan


# ── 1) Consecutive education-record pairs ──────────────────────────────────────
edu2 = edu.copy()
edu2["record_date"] = pd.to_datetime(edu2["record_date"], errors="coerce")
edu2["progress_percent"] = pd.to_numeric(edu2["progress_percent"], errors="coerce")
edu2 = edu2.dropna(subset=["record_date", "progress_percent"])
edu2 = edu2.sort_values(["resident_id", "record_date", "education_record_id"]).reset_index(drop=True)

pair_rows = []
for rid, grp in edu2.groupby("resident_id", sort=False):
    grp = grp.sort_values("record_date").reset_index(drop=True)
    for i in range(len(grp) - 1):
        pair_rows.append({
            "resident_id": rid,
            "record_date": grp.loc[i, "record_date"],
            "current_progress": grp.loc[i, "progress_percent"],
            "edu_education_level": grp.loc[i, "education_level"],
            "next_progress": grp.loc[i + 1, "progress_percent"],   # TARGET
            # Momentum: gain since the previous check-in (NaN for first record;
            # median-imputed by pipeline). Strong forward-looking signal.
            "progress_delta": (
                grp.loc[i, "progress_percent"] - grp.loc[i - 1, "progress_percent"]
            ) if i > 0 else np.nan,
            # Engagement depth: how many prior check-ins has she had?
            "n_prior_edu_records": i,
        })

frame = pd.DataFrame(pair_rows)
print(f"Record pairs (rows): {len(frame)}  across {frame['resident_id'].nunique()} residents")

# ── 2) Static resident features ────────────────────────────────────────────────
# Only truly static fields are joined. Dynamic fields (current_risk_level,
# case_status, reintegration_status) are excluded: joining them as a snapshot
# leaks future case-management decisions into early check-in rows.
RESIDENT_DROP = ["notes_restricted", "initial_case_assessment", "referring_agency_person"]
res2 = res.drop(columns=[c for c in RESIDENT_DROP if c in res.columns], errors="ignore").copy()
res2["present_age_years"] = res2["present_age"].map(parse_years_months)
res2["age_upon_admission_years"] = res2["age_upon_admission"].map(parse_years_months)
res2["date_of_admission"] = pd.to_datetime(res2["date_of_admission"], errors="coerce")

STATIC_COLS = [
    "resident_id", "date_of_admission",
    "present_age_years", "age_upon_admission_years",
    "has_special_needs",
    "case_category", "initial_risk_level",
    "referral_source", "safehouse_id",
]
STATIC_COLS = [c for c in STATIC_COLS if c in res2.columns]
frame = frame.merge(res2[STATIC_COLS], on="resident_id", how="left")
frame["days_since_admission"] = (
    frame["record_date"] - frame["date_of_admission"]
).dt.days.clip(lower=0)

# ── 3) Temporally windowed health aggregates ───────────────────────────────────
# Only health records on or before each row's record_date are included.
# bmi excluded: noisy, sparse, and indirect for education outcomes.
# All three checkup-done rates are now included (previously only psychological was).
hw2 = hw.copy()
hw2["hw_date"] = pd.to_datetime(hw2["record_date"], errors="coerce")
hw_num = ["general_health_score", "nutrition_score", "sleep_quality_score", "energy_level_score"]
hw_bool = ["medical_checkup_done", "dental_checkup_done", "psychological_checkup_done"]
hw_keep = [c for c in hw_num + hw_bool if c in hw2.columns]

hw_cross = (
    frame[["resident_id", "record_date"]]
    .merge(hw2[["resident_id", "hw_date"] + hw_keep], on="resident_id", how="left")
)
hw_cross = hw_cross[hw_cross["hw_date"] <= hw_cross["record_date"]]
hw_num_exist = [c for c in hw_num if c in hw_cross.columns]
hw_bool_exist = [c for c in hw_bool if c in hw_cross.columns]
hw_agg_num = (hw_cross.groupby(["resident_id", "record_date"])[hw_num_exist]
              .mean().rename(columns={c: f"hw_mean_{c}" for c in hw_num_exist}))
hw_agg_bool = (hw_cross.groupby(["resident_id", "record_date"])[hw_bool_exist]
               .mean().rename(columns={c: f"hw_rate_{c}" for c in hw_bool_exist}))
frame = frame.merge(hw_agg_num.reset_index(), on=["resident_id", "record_date"], how="left")
frame = frame.merge(hw_agg_bool.reset_index(), on=["resident_id", "record_date"], how="left")

# ── 4) Temporally windowed incident aggregates ─────────────────────────────────
inc2 = incidents.copy()
inc2["inc_date"] = pd.to_datetime(inc2["incident_date"], errors="coerce")
inc2["severity"] = inc2["severity"].astype(str).str.lower()
inc2["is_high_sev"] = inc2["severity"].isin(["high", "critical", "severe"]).astype(int)
inc2["is_unresolved"] = (~inc2["resolved"].fillna(False).astype(bool)).astype(int)

inc_cross = (
    frame[["resident_id", "record_date"]]
    .merge(inc2[["resident_id", "inc_date", "is_high_sev", "is_unresolved"]], on="resident_id", how="left")
)
inc_cross = inc_cross[inc_cross["inc_date"] <= inc_cross["record_date"]]
inc_cross["days_before"] = (inc_cross["record_date"] - inc_cross["inc_date"]).dt.days

inc_agg = inc_cross.groupby(["resident_id", "record_date"]).agg(
    n_incidents=("is_high_sev", "count"),
    incident_high_rate=("is_high_sev", "mean"),
    incident_unresolved_rate=("is_unresolved", "mean"),
)
# Recent disruption: incidents in the last 90 days carry more signal than
# a lifetime count that dilutes recent events with distant history.
inc_agg_90d = (
    inc_cross[inc_cross["days_before"] <= 90]
    .groupby(["resident_id", "record_date"])
    .size()
    .reset_index(name="n_incidents_last_90d")
)
frame = frame.merge(inc_agg.reset_index(), on=["resident_id", "record_date"], how="left")
frame = frame.merge(inc_agg_90d, on=["resident_id", "record_date"], how="left")
frame["n_incidents"] = frame["n_incidents"].fillna(0)
frame["incident_high_rate"] = frame["incident_high_rate"].fillna(0.0)
frame["incident_unresolved_rate"] = frame["incident_unresolved_rate"].fillna(0.0)
frame["n_incidents_last_90d"] = frame["n_incidents_last_90d"].fillna(0)

# ── 5) Temporally windowed home-visitation aggregates ─────────────────────────
vis2 = visits.copy()
vis2["vis_date"] = pd.to_datetime(vis2["visit_date"], errors="coerce")
vis2["safety_concern"] = vis2["safety_concerns_noted"].fillna(False).astype(bool).astype(int)
vis2["followup"] = vis2["follow_up_needed"].fillna(False).astype(bool).astype(int)

vis_cross = (
    frame[["resident_id", "record_date"]]
    .merge(vis2[["resident_id", "vis_date", "safety_concern", "followup"]], on="resident_id", how="left")
)
vis_cross = vis_cross[vis_cross["vis_date"] <= vis_cross["record_date"]]
vis_agg = vis_cross.groupby(["resident_id", "record_date"]).agg(
    n_home_visitations=("safety_concern", "count"),
    safety_concern_rate=("safety_concern", "mean"),
    followup_needed_rate=("followup", "mean"),
)
frame = frame.merge(vis_agg.reset_index(), on=["resident_id", "record_date"], how="left")
frame["n_home_visitations"] = frame["n_home_visitations"].fillna(0)
frame["safety_concern_rate"] = frame["safety_concern_rate"].fillna(0.0)
frame["followup_needed_rate"] = frame["followup_needed_rate"].fillna(0.0)

# ── 6) Temporally windowed process-session aggregates ────────────────────────
proc2 = proc.copy()
proc2["proc_date"] = pd.to_datetime(proc2["session_date"], errors="coerce")
proc2["concern"] = proc2["concerns_flagged"].fillna(False).astype(bool).astype(int)
proc2["referral"] = proc2["referral_made"].fillna(False).astype(bool).astype(int)

proc_cross = (
    frame[["resident_id", "record_date"]]
    .merge(proc2[["resident_id", "proc_date", "concern", "referral"]], on="resident_id", how="left")
)
proc_cross = proc_cross[proc_cross["proc_date"] <= proc_cross["record_date"]]
proc_agg = proc_cross.groupby(["resident_id", "record_date"]).agg(
    n_process_sessions=("concern", "count"),
    concerns_flagged_rate=("concern", "mean"),
    referral_made_rate=("referral", "mean"),
)
frame = frame.merge(proc_agg.reset_index(), on=["resident_id", "record_date"], how="left")
frame["n_process_sessions"] = frame["n_process_sessions"].fillna(0)
frame["concerns_flagged_rate"] = frame["concerns_flagged_rate"].fillna(0.0)
frame["referral_made_rate"] = frame["referral_made_rate"].fillna(0.0)

# ── 7) Intervention-plan count (windowed on updated_at) ───────────────────────
plans2 = plans.copy()
plans2["plan_date"] = pd.to_datetime(plans2["updated_at"], errors="coerce")
plan_cross = (
    frame[["resident_id", "record_date"]]
    .merge(plans2[["resident_id", "plan_date"]], on="resident_id", how="left")
)
plan_cross = plan_cross[plan_cross["plan_date"] <= plan_cross["record_date"]]
plan_agg = plan_cross.groupby(["resident_id", "record_date"]).size().reset_index(name="n_intervention_plans")
frame = frame.merge(plan_agg, on=["resident_id", "record_date"], how="left")
frame["n_intervention_plans"] = frame["n_intervention_plans"].fillna(0)

# ── 8) Safehouse context (static) ─────────────────────────────────────────────
# province excluded: too granular for ~60 residents; region + safehouse_id cover it.
safe_keep = ["safehouse_id", "region", "capacity_girls", "current_occupancy"]
safe_keep = [c for c in safe_keep if c in safehouses.columns]
frame = frame.merge(safehouses[safe_keep], on="safehouse_id", how="left", suffixes=("", "_sh"))
if "capacity_girls" in frame.columns and "current_occupancy" in frame.columns:
    frame["occupancy_ratio"] = frame["current_occupancy"] / frame["capacity_girls"].replace(0, np.nan)

# ── 9) Windowed attendance rate ───────────────────────────────────────────────
# attendance_rate was previously excluded as a "contemporaneous leaky outcome."
# That concern applies only to using the raw column directly. Aggregating it as a
# windowed mean (mean over all edu records ≤ record_date[i]) follows the same
# temporal contract as every other windowed feature and avoids leakage.
if "attendance_rate" in edu2.columns:
    att_cross = (
        frame[["resident_id", "record_date"]]
        .merge(
            edu2[["resident_id", "record_date", "attendance_rate"]].rename(
                columns={"record_date": "edu_date"}
            ),
            on="resident_id", how="left",
        )
    )
    att_cross = att_cross[att_cross["edu_date"] <= att_cross["record_date"]]
    att_agg = (
        att_cross.groupby(["resident_id", "record_date"])["attendance_rate"]
        .mean()
        .reset_index(name="windowed_attendance_rate")
    )
    frame = frame.merge(att_agg, on=["resident_id", "record_date"], how="left")

# ── 10) Seasonal encoding ─────────────────────────────────────────────────────
# Academic calendars have seasonal patterns (school terms, holidays). Cyclical
# sin/cos encoding preserves the circular distance between months (Dec ↔ Jan).
frame["record_month_sin"] = np.sin(2 * np.pi * frame["record_date"].dt.month / 12)
frame["record_month_cos"] = np.cos(2 * np.pi * frame["record_date"].dt.month / 12)

# ── 11) Feature lists ──────────────────────────────────────────────────────────
# Removed vs. original:
#   days_to_next_record     — future info (next check-in date not known at T_i)
#   current_risk_level      — dynamic field; static join leaks future updates
#   case_status             — dynamic field; static join leaks future updates
#   reintegration_status    — dynamic field; static join leaks future updates
#   family_parent_pwd       — near-zero variance / high missingness for this cohort
#   hw_mean_bmi             — noisy and indirect; other health scores cover overlap
#   province                — too granular for ~60 residents; absorbed by region
# Added vs. original:
#   progress_delta          — momentum (current minus previous progress)
#   n_prior_edu_records     — engagement depth (check-in index within resident)
#   n_incidents_last_90d    — recency-weighted disruption (vs. lifetime count)
#   windowed_attendance_rate — attendance history up to T_i (strong leading indicator)
#   hw_rate_medical_checkup_done, hw_rate_dental_checkup_done — already computed, now used
#   record_month_sin/cos    — cyclical seasonal encoding
TRAJ_NUMERIC_FEATURES = [
    "current_progress",
    "progress_delta",
    "n_prior_edu_records",
    "days_since_admission",
    "present_age_years",
    "age_upon_admission_years",
    "has_special_needs",
    "hw_mean_nutrition_score",
    "hw_mean_energy_level_score",
    "hw_mean_sleep_quality_score",
    "hw_mean_general_health_score",
    "hw_rate_psychological_checkup_done",
    "hw_rate_medical_checkup_done",
    "hw_rate_dental_checkup_done",
    "n_incidents",
    "incident_high_rate",
    "incident_unresolved_rate",
    "n_incidents_last_90d",
    "n_home_visitations",
    "safety_concern_rate",
    "followup_needed_rate",
    "n_process_sessions",
    "concerns_flagged_rate",
    "referral_made_rate",
    "n_intervention_plans",
    "windowed_attendance_rate",
    "occupancy_ratio",
    "record_month_sin",
    "record_month_cos",
]
TRAJ_CATEGORICAL_FEATURES = [
    "case_category",
    "initial_risk_level",
    "referral_source",
    "edu_education_level",
    "region",
]

TRAJ_NUMERIC_FEATURES = [c for c in TRAJ_NUMERIC_FEATURES if c in frame.columns]
TRAJ_CATEGORICAL_FEATURES = [c for c in TRAJ_CATEGORICAL_FEATURES if c in frame.columns]
TRAJ_FEATURE_COLUMNS = TRAJ_NUMERIC_FEATURES + TRAJ_CATEGORICAL_FEATURES

X_traj = frame[TRAJ_FEATURE_COLUMNS].copy()
y_traj = frame["next_progress"].copy()
groups = frame["resident_id"].values   # used by GroupKFold

print(f"\nModeling frame: {len(frame)} rows | {frame['resident_id'].nunique()} residents")
print(f"Features: {len(TRAJ_FEATURE_COLUMNS)} ({len(TRAJ_NUMERIC_FEATURES)} numeric, {len(TRAJ_CATEGORICAL_FEATURES)} categorical)")
print(f"Target (next_progress): min={y_traj.min():.1f}  mean={y_traj.mean():.1f}  max={y_traj.max():.1f}")

# Quick correlation screen
num_probe = [c for c in TRAJ_NUMERIC_FEATURES if X_traj[c].notna().sum() >= 30]
corr = (
    pd.Series({c: y_traj.corr(pd.to_numeric(X_traj[c], errors="coerce")) for c in num_probe})
    .dropna()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
print("\nTop numeric correlations with next_progress:")
print(corr.head(10))
print("\nCheckpoint — Phase 3: record-level frame ready.")

Record pairs (rows): 474  across 60 residents

Modeling frame: 474 rows | 60 residents
Features: 34 (29 numeric, 5 categorical)
Target (next_progress): min=0.0  mean=82.5  max=100.0

Top numeric correlations with next_progress:
current_progress                0.919236
windowed_attendance_rate        0.561523
n_prior_edu_records             0.527192
days_since_admission            0.497409
n_process_sessions              0.464541
n_home_visitations              0.425108
hw_mean_energy_level_score      0.355835
hw_mean_general_health_score    0.338240
has_special_needs              -0.261441
hw_mean_nutrition_score         0.241622
dtype: float64

Checkpoint — Phase 3: record-level frame ready.


/Users/gnelman/Documents/BYU/IS JUNIOR CORE/Winter 2026/Case Studies/INTEX II/Source Code/keeper/ml-pipelines/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/gnelman/Documents/BYU/IS JUNIOR CORE/Winter 2026/Case Studies/INTEX II/Source Code/keeper/ml-pipelines/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## Phase 4: Modeling

We compare three regularised regressors predicting `next_progress` with **GroupKFold (5 folds, grouped by `resident_id`)**.

GroupKFold is critical here: all records for a given resident go into the same fold, so the model is always evaluated on residents it has never seen during training. Without this, the model could learn a resident's personal baseline from her earlier records and exploit it in "test" — inflating CV metrics.

Model selection: lowest CV MAE → highest R².

In [4]:
# Phase 4 — regressor comparison and selection (GroupKFold by resident_id)
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_traj_preprocessor():
    numeric_pipe = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, TRAJ_NUMERIC_FEATURES),
            ("cat", categorical_pipe, TRAJ_CATEGORICAL_FEATURES),
        ]
    )


# GroupKFold: all records for a resident go into the same fold
gkf = GroupKFold(n_splits=5)
cv_splits = list(gkf.split(X_traj, y_traj, groups=groups))

neg_mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)
r2_scorer = make_scorer(r2_score)

candidate_regressors = {
    "ridge": Ridge(alpha=10.0),
    "random_forest": RandomForestRegressor(
        random_state=42, n_estimators=300, max_depth=3, min_samples_leaf=5,
    ),
    "gradient_boosting": GradientBoostingRegressor(
        random_state=42, max_depth=2, n_estimators=100,
        learning_rate=0.05, min_samples_leaf=5, subsample=0.8,
    ),
}

rows = []
cv_by_name = {}
for name, reg in candidate_regressors.items():
    pipe = Pipeline(steps=[("prep", build_traj_preprocessor()), ("reg", clone(reg))])
    scores = cross_validate(
        pipe, X_traj, y_traj,
        cv=cv_splits,
        scoring={"neg_mae": neg_mae_scorer, "r2": r2_scorer},
        return_train_score=True,
    )
    cv_by_name[name] = scores
    rows.append({
        "model": name,
        "cv_mae": -scores["test_neg_mae"].mean(),
        "cv_r2": scores["test_r2"].mean(),
        "train_mae": -scores["train_neg_mae"].mean(),
        "train_r2": scores["train_r2"].mean(),
    })

leaderboard = pd.DataFrame(rows).sort_values(["cv_mae", "cv_r2"], ascending=[True, False])
print("=== Phase 4 CV leaderboard (GroupKFold, n_splits=5) ===")
display(leaderboard.to_string(index=False))

best_name = leaderboard.iloc[0]["model"]
traj_cv_scores = cv_by_name[best_name]
traj_model_name = best_name
traj_pipeline = Pipeline(
    steps=[("prep", build_traj_preprocessor()), ("reg", clone(candidate_regressors[best_name]))]
)
traj_pipeline.fit(X_traj, y_traj)

print(f"\nSelected model: {traj_model_name}")
print(f"CV MAE:  {-traj_cv_scores['test_neg_mae'].mean():.2f} pts  (mean target = {y_traj.mean():.1f})")
print(f"CV R²:   {traj_cv_scores['test_r2'].mean():.3f}")

=== Phase 4 CV leaderboard (GroupKFold, n_splits=5) ===


'            model   cv_mae    cv_r2  train_mae  train_r2\n    random_forest 5.725109 0.820828   4.813834  0.888095\ngradient_boosting 5.905171 0.819280   4.337299  0.913177\n            ridge 7.287698 0.748562   5.559806  0.868767'


Selected model: random_forest
CV MAE:  5.73 pts  (mean target = 82.5)
CV R²:   0.821


## Phase 5: Evaluation

OOF performance of the selected regressor (GroupKFold, same splits as Phase 4), plus:

- Per-resident roster: mean predicted `next_progress`, formatted as percentages with plain-English risk labels, sorted ascending — lowest predicted progress at top
- Feature importance / coefficient ranking

In [5]:
# Phase 5 — OOF evaluation + per-resident roster + feature signals
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_predict

# OOF predictions — same GroupKFold splits, so no resident leaks across folds
pred_oof = cross_val_predict(traj_pipeline, X_traj, y_traj, cv=cv_splits)

mae_oof = mean_absolute_error(y_traj, pred_oof)
r2_oof = r2_score(y_traj, pred_oof)

print("=== Phase 5 OOF performance ===")
print(f"Selected model: {traj_model_name}")
print(f"MAE:  {mae_oof:.2f} pts  (mean next_progress = {y_traj.mean():.1f})")
print(f"R²:   {r2_oof:.3f}")


# ── Risk-label helper (reuse this logic in FastAPI) ────────────────────────────
def format_progress(score: float) -> str:
    """Return a percentage string rounded to one decimal, e.g. '72.3%'."""
    return f"{score:.1f}%"


def risk_label(predicted_next_progress: float, at_risk_threshold: float) -> str:
    """Plain-English risk label for a predicted next progress score."""
    if predicted_next_progress <= at_risk_threshold:
        return "At Risk"
    return "On Track"


# ── Per-resident summary roster ────────────────────────────────────────────────
roster_raw = frame[["resident_id", "current_progress", "next_progress"]].copy()
roster_raw["predicted_next_progress"] = pred_oof

per_res_raw = (
    roster_raw.groupby("resident_id")
    .agg(
        mean_actual_next=("next_progress", "mean"),
        mean_predicted_next=("predicted_next_progress", "mean"),
        n_check_ins=("current_progress", "count"),
    )
    .sort_values("mean_predicted_next")
    .reset_index()
)

at_risk_threshold = per_res_raw["mean_predicted_next"].quantile(0.25)

# Build human-readable roster
roster = pd.DataFrame({
    "Resident ID": per_res_raw["resident_id"],
    "Predicted Next Progress": per_res_raw["mean_predicted_next"].map(format_progress),
    "Actual Next Progress": per_res_raw["mean_actual_next"].map(format_progress),
    "Check-ins": per_res_raw["n_check_ins"].astype(int),
    "Risk Level": per_res_raw["mean_predicted_next"].map(
        lambda v: risk_label(v, at_risk_threshold)
    ),
})

print(f"\n--- Resident Roster (sorted by predicted next progress, ascending) ---")
print(f"At-risk threshold (bottom quartile): {format_progress(at_risk_threshold)}\n")
display(roster.reset_index(drop=True))

print(f"\nAt Risk: {(roster['Risk Level'] == 'At Risk').sum()} residents  |  "
      f"On Track: {(roster['Risk Level'] == 'On Track').sum()} residents")


# ── Feature signals ────────────────────────────────────────────────────────────
prep = traj_pipeline.named_steps["prep"]
reg = traj_pipeline.named_steps["reg"]
feat_names = prep.get_feature_names_out()

if hasattr(reg, "feature_importances_"):
    signal = pd.Series(reg.feature_importances_, index=feat_names).sort_values(ascending=False)
    metric_label = "importance"
elif hasattr(reg, "coef_"):
    signal = pd.Series(reg.coef_, index=feat_names).sort_values(key=lambda s: s.abs(), ascending=False)
    metric_label = "coefficient"
else:
    signal = None
    metric_label = None

if signal is not None:
    print(f"\nTop 15 feature signals ({metric_label}) from model fit on full data:")
    display(signal.head(15).to_frame(metric_label))

=== Phase 5 OOF performance ===
Selected model: random_forest
MAE:  5.73 pts  (mean next_progress = 82.5)
R²:   0.838

--- Resident Roster (sorted by predicted next progress, ascending) ---
At-risk threshold (bottom quartile): 76.9%



,Resident ID,Predicted Next Progress,Actual Next Progress,Check-ins,Risk Level
0,9,40.6%,21.2%,8,At Risk
1,52,44.4%,31.6%,6,At Risk
2,1,52.7%,47.0%,5,At Risk
3,19,57.2%,59.2%,5,At Risk
4,44,57.4%,48.2%,8,At Risk
5,10,58.4%,52.3%,7,At Risk
6,12,63.8%,62.3%,11,At Risk
7,33,66.2%,67.0%,7,At Risk
8,5,71.1%,65.5%,8,At Risk
9,11,71.3%,71.9%,8,At Risk



At Risk: 15 residents  |  On Track: 45 residents

Top 15 feature signals (importance) from model fit on full data:


,importance
num__current_progress,0.987031
num__hw_mean_general_health_score,0.002338
num__windowed_attendance_rate,0.001559
num__hw_mean_energy_level_score,0.001144
num__hw_rate_dental_checkup_done,0.000793
num__hw_rate_psychological_checkup_done,0.000779
cat__edu_education_level_Vocational,0.000742
num__hw_mean_sleep_quality_score,0.000634
num__days_since_admission,0.000628
num__present_age_years,0.000587


## Phase 6: Deployment

Save the trajectory regressor **and** the `at_risk_threshold` together in one artifact so FastAPI has a consistent threshold at inference time without recomputing it per request.

Artifact: `pipelines/girls_education_trajectory_pipeline_v1.sav`

Load in FastAPI with:
```python
artifact = joblib.load("girls_education_trajectory_pipeline_v1.sav")
pipeline, threshold = artifact["pipeline"], artifact["at_risk_threshold"]
```

In [6]:
# Phase 6 — save artifact (pipeline + threshold) and final recommendations
import joblib
from pathlib import Path

_repo = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "Dataset").is_dir()),
    None,
)
if _repo is None:
    raise FileNotFoundError("Could not find Dataset/ — open Jupyter from inside the ml-pipelines repo.")

ARTIFACT_PATH = _repo / "pipelines" / "girls_education_trajectory_pipeline_v1.sav"
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Bundle pipeline + at_risk_threshold so FastAPI uses a consistent cutoff
artifact = {
    "pipeline": traj_pipeline,
    "at_risk_threshold": at_risk_threshold,   # defined in Phase 5
}
joblib.dump(artifact, ARTIFACT_PATH)

# Verify reload
loaded = joblib.load(ARTIFACT_PATH)
_reload_ok = np.allclose(
    loaded["pipeline"].predict(X_traj),
    traj_pipeline.predict(X_traj),
    atol=1e-6,
)

print(f"Saved artifact: {ARTIFACT_PATH}")
print(f"Reload check:   {bool(_reload_ok)}")
print(f"Selected model: {traj_model_name}")
print(f"CV MAE: {-traj_cv_scores['test_neg_mae'].mean():.2f} pts  |  CV R²: {traj_cv_scores['test_r2'].mean():.3f}")
print(f"At-risk threshold bundled: {format_progress(at_risk_threshold)}")

print("\nTop feature signals from selected model (from Phase 5 evaluation):")
try:
    print(signal.head(15).to_string())
except NameError:
    print("(signal not available — re-run Phase 5 first)")

print("\nCRISP-DM complete: Phases 1–6.")

Saved artifact: /Users/gnelman/Documents/BYU/IS JUNIOR CORE/Winter 2026/Case Studies/INTEX II/Source Code/keeper/ml-pipelines/pipelines/girls_education_trajectory_pipeline_v1.sav
Reload check:   True
Selected model: random_forest
CV MAE: 5.73 pts  |  CV R²: 0.821
At-risk threshold bundled: 76.9%

Top feature signals from selected model (from Phase 5 evaluation):
num__current_progress                      0.987031
num__hw_mean_general_health_score          0.002338
num__windowed_attendance_rate              0.001559
num__hw_mean_energy_level_score            0.001144
num__hw_rate_dental_checkup_done           0.000793
num__hw_rate_psychological_checkup_done    0.000779
cat__edu_education_level_Vocational        0.000742
num__hw_mean_sleep_quality_score           0.000634
num__days_since_admission                  0.000628
num__present_age_years                     0.000587
num__concerns_flagged_rate                 0.000405
num__has_special_needs                     0.000397
cat__referr

## Pipeline summary

This notebook builds a **longitudinal education trajectory regressor** on data under `Dataset/lighthouse_csv_v7/`: residents, education check-ins, health, incidents, home visitations, process recordings, intervention plans, and safehouses.

**Goal:** At each education record date **T**, predict **`next_progress`** — the resident's **`progress_percent` at the following check-in (T+1)** — so case teams can prioritize girls whose **predicted next score** is low relative to peers. The problem is **record-level** (~**474** consecutive-record pairs across **60** residents), not one row per resident, to keep real **forward prediction** (features known at T → outcome at T+1) and enough rows for stabler CV.

**Flow:**

1. **Data understanding (Phase 2)** — Load and size the tables listed above; confirm grain and coverage.
2. **Data preparation (Phase 3)** — For each resident, sort `education_records` by date and emit one row per adjacent pair: **`current_progress`**, **`edu_education_level`**, and target **`next_progress`**. Join **static** resident and safehouse fields (only truly immutable fields — `initial_risk_level`, `case_category`, `referral_source` — dynamic fields such as `current_risk_level`, `case_status`, and `reintegration_status` are excluded to avoid leaking future case-management state into early check-in rows). Compute **temporally windowed** aggregates (health means/rates, incident counts/rates, visitation counts/rates, process-session counts/rates, intervention-plan counts, windowed attendance rate) using only events **on or before** that row's **`record_date`** (no future leakage). **`TRAJ_FEATURE_COLUMNS`** = **29** numeric + **5** categorical columns. **`groups`** = **`resident_id`** for grouped CV.

   **Key features added vs. prior version:**
   - `progress_delta` — momentum: gain since the previous check-in (NaN for first record, median-imputed)
   - `n_prior_edu_records` — engagement depth: how many prior check-ins before this row
   - `windowed_attendance_rate` — mean attendance rate over all education records ≤ T_i (strong leading indicator; previously excluded as "leaky" but is valid when windowed)
   - `n_incidents_last_90d` — recent disruption signal (last 90 days vs. lifetime count)
   - `hw_rate_medical_checkup_done`, `hw_rate_dental_checkup_done` — were computed but unused; now included
   - `record_month_sin` / `record_month_cos` — cyclical seasonal encoding (school-term effects)

   **Features removed vs. prior version:**
   - `days_to_next_record` — leakage: the gap to the *next* check-in is not known at inference time
   - `current_risk_level`, `case_status`, `reintegration_status` — soft leakage: dynamic fields whose current snapshot reflects future updates relative to early rows
   - `hw_mean_bmi` — noisy and indirect; other health scores cover the variance
   - `province` — too granular for ~60 residents; absorbed by `region`
   - `family_parent_pwd` — near-zero variance / high missingness for this cohort

3. **Modeling (Phase 4)** — Compare **Ridge**, **RandomForestRegressor**, and **GradientBoostingRegressor** inside a sklearn **`Pipeline`** with **`ColumnTransformer`** (median impute + **`StandardScaler`** on numerics; mode + **`OneHotEncoder`** on categoricals). Validation uses precomputed **`GroupKFold` (5 splits)** splits so **no resident appears in both train and test** — avoiding inflated scores from memorizing individual baselines. Select the model with **lowest mean CV MAE** (tie-break: higher CV **R²**), then **refit on all rows**.
4. **Evaluation (Phase 5)** — **Out-of-fold** predictions with the same splits; MAE/R² in percentage-point units; **per-resident roster** (mean predicted vs actual next progress, check-in counts). **`at_risk_threshold`** = **25th percentile** of mean predicted next progress across residents; labels **"At Risk"** / **"On Track"** for reporting (same logic intended for FastAPI).
5. **Deployment (Phase 6)** — Save a **single joblib dict** to **`pipelines/girls_education_trajectory_pipeline_v1.sav`**: **`pipeline`** (fitted `traj_pipeline`) and **`at_risk_threshold`**, so inference uses a **fixed** cutoff without recomputing it per request.

**Using new data:** Rebuild rows with the **same** pairing, windowing, and column schema as Phase 3, then **`artifact["pipeline"].predict(X_new)`**. Compare predictions to **`artifact["at_risk_threshold"]`** for risk labels. Interpret as **decision support**, not causality; survivors in the cohort imply **selection bias** relative to girls who left early.